In [ ]:
!pip install -q gradio transformers torch torchaudio sentencepiece accelerate ffmpeg-python soundfile

In [ ]:
import gradio as gr
from transformers import pipeline
import torch
import tempfile
import os

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, AutoFeatureExtractor
import numpy as np # Added for np.float32

transcriber_model_name = "openai/whisper-small"
feature_extractor = AutoFeatureExtractor.from_pretrained(transcriber_model_name)
tokenizer = AutoTokenizer.from_pretrained(transcriber_model_name)

transcriber = pipeline(
    "automatic-speech-recognition",
    model=transcriber_model_name,
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
    chunk_length_s=30, # Added for potentially more stable processing of longer audio
    stride_length_s=5, # Added
    dtype=torch.float16 # Changed from torch_dtype to dtype to address warning
)

print("Loading Summarization model...")

class CustomSummarizer:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    def __call__(self, text_or_list_of_texts, **kwargs):
        if isinstance(text_or_list_of_texts, str):
            texts = [text_or_list_of_texts]
        else:
            texts = text_or_list_of_texts

        results = []
        for text in texts:
            inputs = self.tokenizer([text], max_length=1024, return_tensors="pt", truncation=True)
            summary_ids = self.model.generate(
                inputs["input_ids"],
                num_beams=kwargs.get("num_beams", 4),
                max_length=kwargs.get("max_length", 150),
                early_stopping=True
            )
            results.append({"summary_text": self.tokenizer.decode(summary_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)})
        return results

summarizer = CustomSummarizer("facebook/bart-large-cnn")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [ ]:
import torchaudio
import soundfile as sf

def summarize_meeting(audio_filepath):
    try:
        print("Transcribing audio...")

        # Load audio using soundfile
        array, sampling_rate = sf.read(audio_filepath)
        # Ensure float32 type and squeeze to 1D array
        array = array.astype(np.float32).squeeze()

        # Check and resample to 16kHz if necessary
        if sampling_rate != 16000:
            from torchaudio.transforms import Resample
            resampler = Resample(orig_freq=sampling_rate, new_freq=16000)

            # Convert to torch tensor, resample, and convert back to numpy
            # soundfile reads mono as (samples,), stereo as (samples, channels)
            # torchaudio expects (channels, samples)
            # Squeeze to ensure 1D array after all conversions
            if len(array.shape) > 1 and array.shape[1] > 1: # Stereo
                audio_tensor = torch.from_numpy(array).float().T # (channels, samples)
                audio_tensor = resampler(audio_tensor)
                array = torch.mean(audio_tensor, dim=0).numpy().astype(np.float32).squeeze() # Convert to mono, numpy, float32, and squeeze
            else: # Mono
                audio_tensor = torch.from_numpy(array).float().unsqueeze(0) # (1, samples)
                audio_tensor = resampler(audio_tensor)
                array = audio_tensor.squeeze().numpy().astype(np.float32).squeeze() # Convert to numpy, float32, and squeeze
            sampling_rate = 16000 # Update sampling rate after resampling

        # The pipeline expects raw audio input as a dictionary with 'array' and 'sampling_rate'
        audio_input_dict = {
            "array": array,
            "sampling_rate": sampling_rate
        }

        transcript_result = transcriber(audio_input_dict)
        transcript = transcript_result['text']
        print("Transcription complete.")

        print("Summarizing transcript...")
        summary = summarizer(transcript)[0]["summary_text"]
        print("Summarization complete.")

        return transcript, summary

    except Exception as e:
        error_message = f"An error occurred: {e}"
        print(error_message)
        return error_message, error_message

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🎙️ AI Meeting Summarizer

    Upload meeting audio and get:
    - 📝 Transcript
    - ✨ AI Summary

    Powered by Hugging Face 🤗
    """)

    audio_input = gr.Audio(
        type="filepath",
        label="Upload Audio File"
    )

    submit_btn = gr.Button("Generate Summary")

    transcript_output = gr.Textbox(
        label="Transcript",
        lines=15
    )

    summary_output = gr.Textbox(
        label="Meeting Summary",
        lines=10
    )

    submit_btn.click(
        fn=summarize_meeting,
        inputs=audio_input,
        outputs=[
            transcript_output,
            summary_output
        ]
    )

(demo.launch(share=True))

/tmp/ipykernel_3441/4186630175.py:53: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b7a6c8c914f5ad11f9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
